In [1]:
import sys
from pathlib import Path
project_root = Path(__file__).resolve().parents[0] if '__file__' in globals() else Path().resolve().parents[0]
sys.path.insert(0, str(project_root))

In [2]:
from tomato.database import TDAManager
tm = TDAManager()
tm.generate_all()

c:\Users\Clint\OneDrive\Documents\codegolf\tomato-tda\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


▁▁ TDAManager GENERATE-ALL ▁▁
{'datasets': ['bert_strat800', 'bow_strat800', 'tfidf_strat800'],
 'reductions': ['pooled'],
 'metrics': ['cos',
             'cos_ws01',
             'cos_ws01_5nn',
             'cos_ws01_10nn',
             'cos_5nn',
             'cos_10nn',
             'mp1',
             'mp1_ws01',
             'mp1_ws01_5nn',
             'mp1_ws01_10nn',
             'mp1_5nn',
             'mp1_10nn',
             'mpinf',
             'mpinf_ws01',
             'mpinf_ws01_5nn',
             'mpinf_ws01_10nn',
             'mpinf_5nn',
             'mpinf_10nn'],
 'splits': ['full',
            'Dennis Schwartz',
            'Roger Ebert',
            'PG',
            'R',
            'Comedy',
            'Drama',
            'Fresh',
            'Rotten']}

df_critic                                                    ➜    3.4s
df_movie                                                     ➜    0.3s
df_full                                                      ➜

In [3]:
dct = tm._memo;

In [4]:
keys = dct.keys();
full_keys = [k for k in keys if isinstance(k, tuple) and k[-1] == 'full']

In [10]:
import pandas as pd
import numpy as np

def total_persistence(dgms, dims=None):
    """Sum of (death−birth) over given homology dims (or all dims if dims=None),
    skipping any k ≥ len(dgms)."""
    if dims is None:
        dims = range(len(dgms))
    tot = 0.0
    for k in dims:
        if k >= len(dgms):
            continue
        births, deaths = dgms[k][:,0], dgms[k][:,1]
        mask = np.isfinite(deaths)
        tot += (deaths[mask] - births[mask]).sum()
    return tot

def persistent_entropy(dgms, dims=None):
    """Entropy of lifetimes in given dims, skipping missing dims."""
    if dims is None:
        dims = range(len(dgms))
    lifetimes = []
    for k in dims:
        if k >= len(dgms):
            continue
        b, d = dgms[k][:,0], dgms[k][:,1]
        mask = np.isfinite(d)
        Lk = d[mask] - b[mask]
        if Lk.size:
            lifetimes.append(Lk)
    if not lifetimes:
        return 0.0
    L = np.concatenate(lifetimes)
    S = L.sum()
    if S <= 0:
        return 0.0
    p = L / S
    return -np.sum(p * np.log(p))

# define which dim-sets you care about
dims_map = {
    '0':   [0],
    '01':  [0, 1],
    '012':[0, 1, 2],
    '12':  [1, 2],
}

# map short names → summary functions
metrics = {
    'tp':      total_persistence,
    'entropy': persistent_entropy
}

records = []
for key in full_keys:
    # unwrap ripser output
    out = tm.get(*key)
    out = out.item() if isinstance(out, np.ndarray) else out
    dgms = out['dgms']

    rec = dict(zip(['dataset','red','metric','split'], key))
    # for each dim-subset and each summary, compute and stash
    for suffix, dims in dims_map.items():
        for mname, fn in metrics.items():
            rec[f'{mname}_{suffix}'] = fn(dgms, dims)
    records.append(rec)

pd.DataFrame.from_records(records) \
  .to_csv('tda_summary.csv', index=False)

In [6]:
class_dict = {'critic_name': ['Dennis Schwartz', 'Roger Ebert'], 'content_rating': ['PG', 'R'], 'drama_or_comedy': ['Comedy', 'Drama'], 'review_type': ['Fresh', 'Rotten']}
keys

dict_keys([('df_critic',), ('df_movie',), ('df_full',), ('bert_strat800', 'encoding'), ('bert_strat800', 'pooled'), ('bert_strat800', 'pooled', 'cos'), ('bert_strat800', 'pooled', 'cos', 'full'), ('bert_strat800', 'pooled', 'cos', 'Dennis Schwartz'), ('bert_strat800', 'pooled', 'cos', 'Roger Ebert'), ('bert_strat800', 'pooled', 'cos', 'PG'), ('bert_strat800', 'pooled', 'cos', 'R'), ('bert_strat800', 'pooled', 'cos', 'Comedy'), ('bert_strat800', 'pooled', 'cos', 'Drama'), ('bert_strat800', 'pooled', 'cos', 'Fresh'), ('bert_strat800', 'pooled', 'cos', 'Rotten'), ('bert_strat800', 'pooled', 'cos_ws01'), ('bert_strat800', 'pooled', 'cos_ws01', 'full'), ('bert_strat800', 'pooled', 'cos_ws01', 'Dennis Schwartz'), ('bert_strat800', 'pooled', 'cos_ws01', 'Roger Ebert'), ('bert_strat800', 'pooled', 'cos_ws01', 'PG'), ('bert_strat800', 'pooled', 'cos_ws01', 'R'), ('bert_strat800', 'pooled', 'cos_ws01', 'Comedy'), ('bert_strat800', 'pooled', 'cos_ws01', 'Drama'), ('bert_strat800', 'pooled', 'cos_

In [ ]:
"""
pairwise_ripser_wasserstein.py
--------------------------------
Compute Wasserstein (or any) distances between *all* 
class-conditioned persistence diagrams already cached in a TDAManager.

The only things you ever touch are
    • CLASS_DICT – tells the code which labels should be paired
    • DIST_FN     – plug in any binary “distance” you like
Everything else is automatic.
"""
from __future__ import annotations
from itertools import combinations
from pathlib      import Path
import pandas as pd
import joblib, tqdm, inspect
from persim import wasserstein                     # persistence-diagram W₁
from tomato.metrics import wasserstein_distances_sinkhorn_parallel as W_sink

def autodist(A, B,
             *,
             emb_ground="minkowski", emb_p=2, emb_reg=0.1
            ) -> float:
    # unwrap 0-d object arrays if you applied the .item() fix
    if isinstance(A, np.ndarray) and A.dtype == object and A.shape == ():
        A, B = A.item(), B.item()

    # persistence‐diagram case
    if isinstance(A, dict) and "dgms" in A:
        return sum(
            wasserstein(dA, dB)     # ← no order=… or internal_p=…
            for dA, dB in zip(A["dgms"], B["dgms"])
        )

    # otherwise fall back to Sinkhorn‐Wasserstein on embeddings…
    def to_df(X):
        if hasattr(X, "columns"):
            return X
        return pd.DataFrame(X, columns=[f"dim_{i}" for i in range(X.shape[1])])
    return W_sink(
        to_df(A), to_df(B),
        ground_metric=emb_ground, p=emb_p, reg=emb_reg, nCores=1
    )

# ---------- USER-TUNABLE ONE-LINERS ---------------------------------
CLASS_DICT = {
    "critic_name"     : ["Dennis Schwartz", "Roger Ebert"],
    "content_rating"  : ["PG", "R"],
    "drama_or_comedy" : ["Comedy", "Drama"],
    "review_type"     : ["Fresh", "Rotten"],
}
DIST_FN  = autodist
N_CORES  = -1
OUT_CSV  = Path("ripser_wasserstein_pairs_p2.csv")
# --------------------------------------------------------------------

def _categorize(label: str) -> str | None:
    """Return the CLASS_DICT key this label belongs to, else None."""
    for cat, vals in CLASS_DICT.items():
        if label in vals: return cat
    return None


def _group_keys(keys) -> dict[tuple[str, str, str], dict[str, tuple]]:
    """
    Return { (dataset,pooled,metric) : {label -> full_key_tuple} }
    for the 4-tuples of interest (len==4 & last != 'full').
    """
    groups: dict[tuple[str, str, str], dict[str, tuple]] = {}
    for k in keys:
        if len(k) != 4 or k[-1] == "full": continue
        triplet, label = k[:3], k[3]
        groups.setdefault(triplet, {})[label] = k
    return groups

def _plan_pairs(label_dict: dict[str,tuple]) -> list[tuple[str,str,tuple,tuple]]:
    """
    Given {label -> key}, return list of (label1,label2,key1,key2)
    restricted to label pairs specified in CLASS_DICT.
    """
    pairs = []
    for cat, vals in CLASS_DICT.items():
        if not all(v in label_dict for v in vals): continue
        l1, l2 = vals
        pairs.append((l1, l2, label_dict[l1], label_dict[l2]))
    return pairs

def _compute_row(tm, triplet, l1, l2, k1, k2):
    """Helper for parallel map – returns a dictionary (one CSV row)."""
    d = DIST_FN(tm.get(*k1), tm.get(*k2))
    ds, pool, metric = triplet
    cat_group = _categorize(l1)
    return {
        "dataset" : ds,
        "pooled"  : pool,
        "metric"  : metric,
        "cat_grp" : cat_group,
        "cat1"    : l1,
        "cat2"    : l2,
        "distance": d,
    }

def run(tm, keys) -> pd.DataFrame:
    """Main entry: supply your TDAManager and its .keys()."""
    groups    = _group_keys(keys)
    tasks = []                                           # ← FIXED BLOCK
    for triplet, lbl_dict in groups.items():
        for l1, l2, k1, k2 in _plan_pairs(lbl_dict):
            tasks.append((triplet, l1, l2, k1, k2))      # flat 5-tuple

    fn_name = getattr(DIST_FN, "__name__", "dist_fn")
    desc    = f"Computing {fn_name} ({len(tasks)} pairs)"

    rows = joblib.Parallel(n_jobs=N_CORES)(              # ← UNPACK MATCHES
        joblib.delayed(_compute_row)(tm, triplet, l1, l2, k1, k2)
        for triplet, l1, l2, k1, k2 in tqdm.tqdm(tasks, desc=desc)
    )
    df = pd.DataFrame(rows)
    df.to_csv(OUT_CSV, index=False)
    return df

df = run(tm, keys)
print(f"✓ wrote {len(df)} rows to {OUT_CSV.resolve()}")

Computing autodist (216 pairs): 100%|██████████| 216/216 [10:48<00:00,  3.00s/it]


✓ wrote 216 rows to C:\Users\Clint\OneDrive\Documents\codegolf\tomato-tda\working\ripser_wasserstein_pairs_p2.csv


In [8]:

from __future__ import annotations
from itertools import combinations
from pathlib      import Path
import joblib, tqdm, inspect             # persistence-diagram W₁
import numpy as np, pandas as pd
from persim import wasserstein, bottleneck, sliced_wasserstein
from persim.landscapes import PersLandscapeExact            # exact landscapes
from persim.images      import PersImage                     # persistence images

from tomato.metrics import wasserstein_distances_sinkhorn_parallel as W_sink

import numpy as np, pandas as pd

def autodist(A, B,
             *,
             emb_ground="minkowski", emb_p=2, emb_reg=0.1
            ) -> float:
    # unwrap 0-d object arrays if you applied the .item() fix
    if isinstance(A, np.ndarray) and A.dtype == object and A.shape == ():
        A, B = A.item(), B.item()

    # persistence‐diagram case
    if isinstance(A, dict) and "dgms" in A:
        return sum(
            wasserstein(dA, dB)     # ← no order=… or internal_p=…
            for dA, dB in zip(A["dgms"], B["dgms"])
        )

    # otherwise fall back to Sinkhorn‐Wasserstein on embeddings…
    def to_df(X):
        if hasattr(X, "columns"):
            return X
        return pd.DataFrame(X, columns=[f"dim_{i}" for i in range(X.shape[1])])
    return W_sink(
        to_df(A), to_df(B),
        ground_metric=emb_ground, p=emb_p, reg=emb_reg, nCores=1
    )

def wd_sum(A, B):        # original behaviour
    return sum(wasserstein(dA, dB) for dA, dB in zip(A["dgms"], B["dgms"]))

def wd_dim(A, B, i):     # per-dimension W₁
    return wasserstein(A["dgms"][i], B["dgms"][i])

def bottleneck_sum(A, B):
    return sum(bottleneck(dA, dB) for dA, dB in zip(A["dgms"], B["dgms"]))

def sw1_sum(A, B, n_dir=50):
    return sum(sliced_wasserstein(dA, dB, n_dir) for dA, dB in zip(A["dgms"], B["dgms"]))

def landscape_L2(A, B, p=2):
    """‖PL(A) − PL(B)‖_2 summed over homology dims."""
    tot = 0.0
    n_dims = min(len(A["dgms"]), len(B["dgms"]))
    for i in range(n_dims):
        # give PersLandscapeExact the FULL list, not a single diagram
        try:
            plA = PersLandscapeExact(dgms=A["dgms"], hom_deg=i)
            plB = PersLandscapeExact(dgms=B["dgms"], hom_deg=i)
            tot += (plA - plB).p_norm(p)
        except (IndexError, ValueError):     # empty diagram or bad index
            continue                         # just skip this dimension
    return tot
def image_L2(A, B, res=50, spread=1.0):
    pim = PersImage(pixels=[res, res], spread=spread)
    imgA = pim.transform(A["dgms"]).reshape(-1, res, res)
    imgB = pim.transform(B["dgms"]).reshape(-1, res, res)
    return np.linalg.norm(imgA - imgB)

# ---------- USER-TUNABLE ONE-LINERS ---------------------------------
CLASS_DICT = {
    "critic_name"     : ["Dennis Schwartz", "Roger Ebert"],
    "content_rating"  : ["PG", "R"],
    "drama_or_comedy" : ["Comedy", "Drama"],
    "review_type"     : ["Fresh", "Rotten"],
}
DIST_FNS = {
    "wd_sum"      : wd_sum,
    "wdim0"       : lambda A,B: wd_dim(A,B,0),
    "wdim1"       : lambda A,B: wd_dim(A,B,1),
    "bottleneck"  : bottleneck_sum,
    "sw1"         : sw1_sum,
    "land_L2"     : landscape_L2,
    "img_L2"      : image_L2,
}
N_CORES  = -1
OUT_CSV  = Path("ripser_pairs_dists.csv")
# --------------------------------------------------------------------

def _categorize(label: str) -> str | None:
    """Return the CLASS_DICT key this label belongs to, else None."""
    for cat, vals in CLASS_DICT.items():
        if label in vals: return cat
    return None


def _group_keys(keys) -> dict[tuple[str, str, str], dict[str, tuple]]:
    """
    Return { (dataset,pooled,metric) : {label -> full_key_tuple} }
    for the 4-tuples of interest (len==4 & last != 'full').
    """
    groups: dict[tuple[str, str, str], dict[str, tuple]] = {}
    for k in keys:
        if len(k) != 4 or k[-1] == "full": continue
        triplet, label = k[:3], k[3]
        groups.setdefault(triplet, {})[label] = k
    return groups

def _plan_pairs(label_dict: dict[str,tuple]) -> list[tuple[str,str,tuple,tuple]]:
    """
    Given {label -> key}, return list of (label1,label2,key1,key2)
    restricted to label pairs specified in CLASS_DICT.
    """
    pairs = []
    for cat, vals in CLASS_DICT.items():
        if not all(v in label_dict for v in vals): continue
        l1, l2 = vals
        pairs.append((l1, l2, label_dict[l1], label_dict[l2]))
    return pairs

def _compute_row(tm, triplet, l1, l2, k1, k2):
    A, B = tm.get(*k1), tm.get(*k2)
    row = {
        "dataset": triplet[0], "pooled": triplet[1], "metric": triplet[2],
        "cat_grp": _categorize(l1),  "cat1": l1,  "cat2": l2,
    }
    for name, fn in DIST_FNS.items():
        row[name] = fn(A, B)
    return row

def run(tm, keys) -> pd.DataFrame:
    groups, tasks = _group_keys(keys), []
    for triplet, lbl_dict in groups.items():
        for l1, l2, k1, k2 in _plan_pairs(lbl_dict):
            tasks.append((triplet, l1, l2, k1, k2))

    # --------- FIX: describe the whole suite, not a single fn ----------
    desc = (f"Computing {len(DIST_FNS)} metrics "
            f"for {len(tasks)} class-pairs")
    # ------------------------------------------------------------------

    rows = joblib.Parallel(n_jobs=N_CORES)(
        joblib.delayed(_compute_row)(tm, *t)           # t already flat-5
        for t in tqdm.tqdm(tasks, desc=desc)
    )
    df = pd.DataFrame(rows).to_csv(OUT_CSV, index=False)
    return df

df = run(tm, keys)
print(f"✓ wrote {len(df)} rows to {OUT_CSV.resolve()}")

AttributeError: 'list' object has no attribute 'reshape'

In [ ]:
"""
Pair-wise TDA­–distance battery
──────────────────────────────
* Emits one column per metric–dimension combo, plus a 3-dim sum (_012)
* Minimal repetition: one registry drives everything
"""
from __future__ import annotations
from itertools import product
from pathlib   import Path

import joblib, tqdm
import numpy as np
import pandas as pd

from persim import wasserstein, bottleneck, sliced_wasserstein
from persim.landscapes import PersLandscapeExact
from persim.images      import PersImage

from tomato.metrics import wasserstein_distances_sinkhorn_parallel as W_sink

def wd_dim(A, B, i):                    # W₁ on diagram i
    return wasserstein(A["dgms"][i], B["dgms"][i])

def bn_dim(A, B, i):                    # bottleneck on diagram i
    return bottleneck(A["dgms"][i], B["dgms"][i])

def sw1_dim(A, B, i, *, dirs=50):       # sliced-W₁ on diagram i
    return sliced_wasserstein(A["dgms"][i], B["dgms"][i], dirs)

def landL2_dim(A, B, i, *, p=2):        # ‖landscape diff‖₂ on dim i
    try:
        plA = PersLandscapeExact(dgms=A["dgms"], hom_deg=i)
        plB = PersLandscapeExact(dgms=B["dgms"], hom_deg=i)
        return (plA - plB).p_norm(p)
    except (IndexError, ValueError):     # dim absent / empty
        return np.nan

def imgL2_dim(A, B, i, *, res=50, spread=1.0):   # ‖image diff‖₂ on dim i
    try:
        pim = PersImage(pixels=[res, res], spread=spread)
        a = np.asarray(pim.transform([A["dgms"][i]])).squeeze()
        b = np.asarray(pim.transform([B["dgms"][i]])).squeeze()
        return np.linalg.norm(a - b)
    except (IndexError, ValueError):
        return np.nan

METRIC_FNS = {         # base-metric → kernel over a single homology dim
    "wd"     : wd_dim,
    "bn"     : bn_dim,
    "sw1"    : sw1_dim,
    "landL2" : landL2_dim,
    "imgL2"  : imgL2_dim,
}
DIMS_MAP = {           # suffix → list of dims to aggregate
    "0"   : [0],
    "1"   : [1],
    "2"   : [2],
    "012" : [0, 1, 2],
}

CLASS_DICT = {         # edit freely
    "critic_name"     : ["Dennis Schwartz", "Roger Ebert"],
    "content_rating"  : ["PG", "R"],
    "drama_or_comedy" : ["Comedy", "Drama"],
    "review_type"     : ["Fresh", "Rotten"],
}
N_CORES  = -1
OUT_CSV  = Path("ripser_pairs_dists.csv")

def _categorize(label: str) -> str | None:
    for cat, vals in CLASS_DICT.items():
        if label in vals: return cat
    return None

def _group_keys(keys) -> dict[tuple[str,str,str], dict[str,tuple]]:
    groups: dict[tuple[str,str,str], dict[str,tuple]] = {}
    for k in keys:                                   # k is a 4-tuple
        if len(k) != 4 or k[-1] == "full": continue
        groups.setdefault(k[:3], {})[k[3]] = k       # (ds,pooled,met)↦label→key
    return groups

def _plan_pairs(lbl2key: dict[str,tuple]):
    for cat, (v1, v2) in CLASS_DICT.items():
        if v1 in lbl2key and v2 in lbl2key:
            yield v1, v2, lbl2key[v1], lbl2key[v2]

def _compute_row(tm, triplet, l1, l2, k1, k2):
    A, B = tm.get(*k1), tm.get(*k2)
    row  = dict(dataset=triplet[0], pooled=triplet[1], metric=triplet[2],
                cat_grp=_categorize(l1), cat1=l1, cat2=l2)
    for mname, fn in METRIC_FNS.items():
        for suff, dims in DIMS_MAP.items():
            vals = [fn(A, B, d) for d in dims]
            row[f"{mname}_{suff}"] = np.nansum(vals) if len(dims) > 1 else vals[0]
    return row

def run(tm, keys) -> pd.DataFrame:
    tasks = [(trip, *pair)
             for trip, lbl2key in _group_keys(keys).items()
             for pair            in _plan_pairs(lbl2key)]
    desc  = f"{len(tasks)} pairs × {len(METRIC_FNS)*len(DIMS_MAP)} metrics"
    rows  = joblib.Parallel(n_jobs=N_CORES)(
        joblib.delayed(_compute_row)(tm, *t)
        for t in tqdm.tqdm(tasks, desc=desc)
    )
    return pd.DataFrame(rows)

df = run(tm, keys)          # tm, keys assumed imported / pre-built
df.to_csv(OUT_CSV, index=False)
print(f"✓ wrote {len(df)} rows → {OUT_CSV.resolve()}")


216 pairs × 20 metrics:  83%|████████▎ | 180/216 [16:28<03:24,  5.67s/it]